# Does multi-goal supervision produce transferable distances?

Select **GPU → Run all**. This experiment trains two new adapters in the existing final **radius-34** region around **(50,42)**.

- **Fresh single-goal control:** every fitting pair references the original goal.
- **Multi-goal adapter:** fitting pairs connect different positions throughout the same region.
- **Historical adapter:** the earlier four-stage single-goal run is evaluated as an additional descriptive comparator.

Both new models start at identity and use **2,000 updates × 32 pairs**, seed 0, Adam lr 1e-4, bottleneck 64, no anchor loss. Both endpoints are adapted. They see the full final region at once; this is not another expanding curriculum.

The **15 shifted goals** from the last sweep are reserved from fitting, including goals at **y=18, y=10 and y=8**. The original goal is a seen-reference control. All earlier experiments are preserved; new outputs use `radial_multigoal_v1`.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, shutil, signal, subprocess, sys, time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE = Path('/content/drive/MyDrive')
SOURCE_RUN = DRIVE / 'datasets/curriculum_cost_adapter/radial_distance_v1/20261106T101715235909Z'
SOURCE_STAGE = 4
assert (SOURCE_RUN/'run.json').is_file(), 'Set SOURCE_RUN to your existing radial_distance_v1 run folder.'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = DRIVE / 'datasets/curriculum_cost_adapter/radial_multigoal_v1' / RUN_ID
CACHE = Path('/content/radial-multigoal-cache') / RUN_ID
REPO = Path('/content/i-jepa-wm-radial-multigoal')
REPO_URL = 'https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT = 'PUBLISH_SOURCE_COMMIT'
DINO_ROOT = Path('/content/dino_wm')
DINO_COMMIT = '0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE = DRIVE / 'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
ENCODER = Path('/content/radial-goal-sweep-artifacts/encoder.pth.tar')
ENCODER_SHA = '3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
assert ENCODER_DRIVE.is_file(), f'Missing encoder: {ENCODER_DRIVE}'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

def shell(command, cwd=None):
    command=list(map(str,command)); print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout: print(line,end='',flush=True)
            returncode=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.terminate()
            raise
    if returncode: raise subprocess.CalledProcessError(returncode,command)

print('GPU:', torch.cuda.get_device_name())
print('New Drive results:', OUTPUT)
print('Historical source run:',SOURCE_RUN,'stage',SOURCE_STAGE)
print('Reserved goals: same 15 near/far shifts as the previous sweep.')
print('Local cache:',CACHE)

## Load the pinned code, encoder and renderer

The existing distance run supplies the radial geometry, historical checkpoint and provenance. Its old local feature cache is unnecessary. New features are encoded once and reused within this run; the encoder stays frozen.

In [ ]:
if not REPO.exists():
    shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
assert SOURCE_COMMIT, 'Use the published notebook with a verified source pin.'
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow'])
if not DINO_ROOT.exists(): shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ENCODER.parent.mkdir(parents=True,exist_ok=True)
if not ENCODER.exists() or sha256(ENCODER)!=ENCODER_SHA: shutil.copy2(ENCODER_DRIVE,ENCODER)
assert sha256(ENCODER)==ENCODER_SHA, 'Unexpected encoder weights.'
print('Ready. Encoder preparation and adapter fitting run in separate processes to release GPU memory.')


def command(operation,branch=None):
    args=[sys.executable,'-m','src.mylonas_ijepa_wm.radial_multigoal_experiment',operation,
          '--output-dir',OUTPUT]
    if operation!='report':args+=['--cache-dir',CACHE,'--device','cuda']
    if operation=='prepare':args+=['--source-run',SOURCE_RUN,'--encoder-checkpoint',ENCODER,'--dino-wm-root',DINO_ROOT]
    if operation=='train':
        if branch not in ('single','multi'):raise ValueError('Choose single or multi branch.')
        args+=['--branch',branch]
    return args

## Prepare fitting nodes and reserve the evaluation goals

Use the same original 115 radial nodes plus the original goal as candidates. Remove candidates whose coordinates **or rendered images** match a reserved goal, then deduplicate identical training images. The exact remaining count is printed below.

Multi-goal pairs connect distinct remaining nodes, with physical distances at most 34. Single-goal pairs use the same remaining radial nodes and the original reference. Targets remain **Euclidean distance / 34**. Reserved goals and heldout grid queries never enter optimization, calibration or model selection.

The local token cache is separate from Drive. Around 500 positions are encoded in float32 batches of eight. Preparation verifies that moving the environment goal does not change fixed-position observations.

In [ ]:
started=time.monotonic()
shell(command('prepare'),cwd=REPO)
config=json.loads((OUTPUT/'config.json').read_text())
design=json.loads((OUTPUT/'design.json').read_text())
print('Fitting configuration:',config['fitting'])
print('Candidate / admitted / removed nodes:',len(design['candidate_ids']),len(design['train_ids']),len(design['removed_candidate_ids']))
print('Single / multi pair populations:',len(design['single_pairs']),len(design['multi_pairs']))
print('Evaluation goals:',[design['points'][i] for i in design['goal_ids']])
print(f'Preparation took {(time.monotonic()-started)/60:.2f} minutes.')

## Train both fresh adapters

For every sampled pair A,B, minimize:

`mean((MSE(adapter(z_A), adapter(z_B)) - distance(A,B)/34)**2)`

Both arms have the same initialization, architecture, update count, pair batch size and optimizer settings. Their endpoint sampling distributions differ by design; counts are saved. The single-goal arm repeatedly uses the original reference; the multi-goal arm distributes references over the fitting nodes.

The printed **fixed training probe** is chosen from each arm's fitting pairs before optimization. It is distinct from noisy minibatch loss and from the later heldout results. There is no early stopping or tuning on test goals.

In [ ]:
started=time.monotonic()
for branch in ('single','multi'):
    print('\nTRAINING',branch,flush=True)
    shell(command('train',branch),cwd=REPO)
    t=json.loads((OUTPUT/'branches'/branch/'training.json').read_text())
    print('Fixed training probe before / after:',t['probe_before'],t['probe_after'])
    print('Sampled pairs / endpoint evaluations:',sum(t['pair_sample_counts']),sum(t['endpoint_counts']))
print(f'Both training arms took {(time.monotonic()-started)/60:.2f} minutes.')

## Evaluate on one shared heldout cohort

Every method and goal uses the same ball positions: inside the original R34 disk, no farther than 34 from every tested goal, and excluding coordinate/image overlap with fitting nodes and evaluation references. Full heatmaps also show points outside this main cohort.

Adapted distance estimate = **34 × adapted MSE**. The raw baseline uses one scalar fitted on **multi-goal training pairs only**, then kept fixed for every goal. This differs from the old single-goal raw calibration; it is labelled and saved separately.

MAE/RMSE are in simulation units. **Uphill** means adapted cost increased on a tested adjacent grid move that physically approached the evaluated goal; ties are counted separately. These are spatial diagnostics, not planning trials. All comparisons are descriptive for this one seed/checkpoint.

In [ ]:
started=time.monotonic()
shell(command('evaluate'),cwd=REPO)
shell(command('report'),cwd=REPO)
r=json.loads((OUTPUT/'report.json').read_text())
from IPython.display import display,Image

def number(value):return 'N/A' if value is None else f'{value:.3f}'
def pct(value):return 'N/A' if value is None else f'{value:.1%}'

print('Source unchanged:',r['source_unchanged'])
print('Cached/fresh original goal:',r['original_goal_consistency'])
print('Raw scalar:',r['raw_calibration'],'| population:',r['raw_calibration_population'])
print('Historical raw scalar (not used for main raw baseline):',r['historical_raw_calibration'])
print('DISTANCE MAE: same heldout queries for every method and goal')
print(f"{'id':>3} {'goal':>19} {'status':>9} {'N':>5} {'raw':>8} {'single':>8} {'multi':>8} {'old':>8}")
for j,g in enumerate(r['methods']['multi']):
    metrics=[r['methods'][b][j]['heldout_common']['adapted']['mae'] for b in ('single','multi','historical')]
    print(f"{j:>3} {str(tuple(round(v,3) for v in g['goal'])):>19} {g['reference_status']:>9} {g['heldout_common']['count']:>5} "
          f"{number(g['heldout_common']['raw_calibrated']['mae']):>8} "+' '.join(f'{number(v):>8}' for v in metrics))
print('\nUPHILL: fraction of tested approaching moves whose cost increases')
print(f"{'id':>3} {'raw':>8} {'single':>8} {'multi':>8} {'old':>8} {'steps':>7} {'multi ties':>11}")
for j,g in enumerate(r['methods']['multi']):
    values=[r['methods'][b][j]['heldout_directional']['adapted_up_fraction'] for b in ('single','multi','historical')]
    print(f"{j:>3} {pct(g['heldout_directional']['raw_up_fraction']):>8} "+' '.join(f'{pct(v):>8}' for v in values)+
          f" {g['heldout_directional']['comparisons']:>7} {g['heldout_directional']['adapted_ties']:>11}")
for name in ('coverage','training_loss','goal_summary'):display(Image(filename=str(OUTPUT/f'{name}.png')))
print(f'Evaluation/report took {(time.monotonic()-started)/60:.2f} minutes.')

## Heatmaps for all goals

Raw has its own scale. **Fresh single-goal, multi-goal and true-distance maps share one full-range scale per goal**, without clipping. Error is in simulation units and has its own scale. White points are admitted fitting nodes; the dashed circle is the original R34 region; the orange star is the evaluated reference.

The historical adapter is included in the numerical comparison. These maps focus on the matched fresh models. Goal 0 is seen; every shifted goal is withheld.

In [ ]:
for j,g in enumerate(r['methods']['multi']):
    print('GOAL',j,g['goal'],'| reference:',g['reference_status'])
    print('Raw / single / multi self costs:',g['raw_self_cost'],
          r['methods']['single'][j]['adapted_self_cost'],g['adapted_self_cost'])
    display(Image(filename=str(OUTPUT/f'goal_{j}_heatmaps.png')))

## Read the withheld goals separately from the seen control

The useful question is whether multi-goal supervision improves **both distance accuracy and directional guidance on reserved goals**, especially the distant ones. Check the original goal too, to see whether that improvement costs accuracy there.

The summary below counts only shifted goals, with nearby and distant groups reported separately. Goals are correlated tests of one trained model, not independent training seeds. Good fitting loss alone does not establish transfer. The historical curriculum used a different schedule and can include some current test-reference images in its old training; it is descriptive only.

In [ ]:
rows=r['methods']['multi']
for title,indices in [('all withheld',list(range(1,len(rows)))),
                      ('nearby withheld',[j for j,g in enumerate(rows) if j and g['goal'][1]>=22]),
                      ('distant withheld (y<22)',[j for j,g in enumerate(rows) if j and g['goal'][1]<22])]:
    valid=[j for j in indices if rows[j]['heldout_common']['adapted']['mae'] is not None]
    if not valid:
        print(title,': N/A (no common heldout queries)');continue
    print('\n',title,':',len(valid),'goals')
    for metric,label in [('mae','distance MAE')]:
        for method in ('single','multi','historical'):
            vals=[r['methods'][method][j]['heldout_common']['adapted'][metric] for j in valid]
            print(method,'mean per-goal',label,number(sum(vals)/len(vals)))
    wins=sum(rows[j]['heldout_common']['adapted']['mae']<r['methods']['single'][j]['heldout_common']['adapted']['mae'] for j in valid)
    directional=[j for j in valid if rows[j]['heldout_directional']['adapted_up_fraction'] is not None]
    print('Multi improves distance MAE over matched single:',wins,'/',len(valid))
    if directional:
        wins=sum(rows[j]['heldout_directional']['adapted_up_fraction']<r['methods']['single'][j]['heldout_directional']['adapted_up_fraction'] for j in directional)
        print('Multi improves uphill fraction over matched single:',wins,'/',len(directional))
print('Original-goal control single/multi MAE:',
      number(r['methods']['single'][0]['heldout_common']['adapted']['mae']),number(rows[0]['heldout_common']['adapted']['mae']))
print('Results:',OUTPUT)

## Return the executed notebook

Download **File → Download → Download .ipynb** and attach it here. Drive retains checkpoints, training tokens, reference tokens/images, pair/split populations, endpoint exposure counts, reports and PNG/PDF figures. Numerical reports and missing figures can be regenerated after the local token cache is gone.

If interrupted during an unfinished fitting step, start a new run timestamp. Completed steps validate inputs/hashes before reuse. Local synthetic verification checks the workflow; actual GPU results determine whether this supervision transfers.